In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import optuna
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from src.utils import get_features_data_path



data_path = get_features_data_path()
print(f"Szukam pliku: {data_path}")
print(f"Plik istnieje: {data_path.exists()}")

df = pd.read_csv(data_path, parse_dates=['timestamp'])
print(f"✓ Wczytano: {df.shape}")


Szukam pliku: C:\Users\aiszk\PycharmProjects\PythonProject\data\processed\features_data.csv
Plik istnieje: True
✓ Wczytano: (4484722, 54)


In [7]:
# Usuń kolumny nieistotne i kategoryczne
drop_cols = ['timestamp', 'meter_id', 'campus_id', 'consumption', 'date_dt', 'id', 'id_building', 'id_campus']
drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(drop_cols, axis=1)
y = df['consumption']

# Usuń kolumny kategoryczne (strings)
categorical_cols = X.select_dtypes(include=['object']).columns
X = X.drop(categorical_cols, axis=1)

# Uzupełnij braki danymi średnimi
X = X.fillna(X.mean())

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split na zbiór treningowy i testowy – użyj temporalnie jeśli chcesz
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


X shape: (4484722, 48)
y shape: (4484722,)


In [8]:
def objective(trial):
    alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
    max_iter = trial.suggest_int('max_iter', 1000, 10000)

    model = Ridge(alpha=alpha, max_iter=max_iter)

    # Walidacja krzyżowa 3-krotna
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return -score


In [9]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print(f"Najlepsze parametry: {study.best_params}")
print(f"Najlepsza wartość RMSE CV: {study.best_value:.4f}")


[I 2025-11-24 23:52:07,438] A new study created in memory with name: no-name-1f490d08-4ec3-4eac-b6a4-e161a31de309
<ipython-input-8-eb92fb3ba844>:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  alpha = trial.suggest_loguniform('alpha', 1e-4, 10.0)
C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=4.48155e-22): result may not be accurate.
  return linalg.solve(A, Xy, sym_pos=True,
C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=2.15608e-23): result may not be accurate.
  return linalg.solve(A, Xy, sym_pos=True,
C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=2.22171e-23): result may not be accurate.
  retur

Najlepsze parametry: {'alpha': 9.86594259995361, 'max_iter': 8521}
Najlepsza wartość RMSE CV: 1.0171


In [10]:
best_params = study.best_params
best_model = Ridge(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE najlepszego modelu: {test_rmse:.4f}")


C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\_ridge.py:147: LinAlgWarning: Ill-conditioned matrix (rcond=9.04834e-20): result may not be accurate.
  return linalg.solve(A, Xy, sym_pos=True,


Test RMSE najlepszego modelu: 1.0613


In [11]:
from pathlib import Path
import os

# Znajdź katalog główny projektu (przykład: rodzic notebooków)
project_root = Path(__file__).parent.parent if "__file__" in locals() else Path.cwd().parent

# Ustaw ścieżkę docelową
models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "ridge_optimized.joblib"

import joblib
joblib.dump(best_model, model_path)
print(f"✓ Model zapisany: {model_path}")

✓ Model zapisany: C:\Users\aiszk\PycharmProjects\PythonProject\models\ridge_optimized.joblib


In [13]:
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score

# Funkcja celu dla XGBoost
def objective_xgb(trial):
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

    }
    model = xgb.XGBRegressor(**param, random_state=42, n_jobs=-1)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return -score


# Optymalizacja XGBoost
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=50)

print(f"Najlepsze parametry XGBoost: {study_xgb.best_params}")
print(f"Najlepsze RMSE CV XGBoost: {study_xgb.best_value:.4f}")

# Trening z najlepszymi parametrami
best_params_xgb = study_xgb.best_params
best_model_xgb = xgb.XGBRegressor(**best_params_xgb, random_state=42, n_jobs=-1)
best_model_xgb.fit(X_train, y_train)

# Ewaluacja na zbiorze testowym
y_pred_xgb = best_model_xgb.predict(X_test)
from sklearn.metrics import mean_squared_error
import numpy as np
test_rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
print(f"Test RMSE XGBoost: {test_rmse_xgb:.4f}")


[I 2025-11-25 00:01:44,841] A new study created in memory with name: no-name-1c981b2b-af28-4893-88c8-2f42aead53b3
[I 2025-11-25 00:07:08,435] Trial 0 finished with value: 5.615414212648116 and parameters: {'n_estimators': 235, 'max_depth': 7, 'learning_rate': 0.2706137908403674, 'subsample': 0.5654847599681516, 'colsample_bytree': 0.5071765456111457}. Best is trial 0 with value: 5.615414212648116.
[I 2025-11-25 00:09:29,743] Trial 1 finished with value: 4.812455156299205 and parameters: {'n_estimators': 89, 'max_depth': 9, 'learning_rate': 0.035813706133271374, 'subsample': 0.7920371161834958, 'colsample_bytree': 0.908963362019106}. Best is trial 1 with value: 4.812455156299205.
[I 2025-11-25 00:14:20,010] Trial 2 finished with value: 4.897598902157413 and parameters: {'n_estimators': 233, 'max_depth': 8, 'learning_rate': 0.08068745167332003, 'subsample': 0.6789391631407227, 'colsample_bytree': 0.8170734207676371}. Best is trial 1 with value: 4.812455156299205.
[I 2025-11-25 00:20:06,5

Najlepsze parametry XGBoost: {'n_estimators': 77, 'max_depth': 3, 'learning_rate': 0.1765554284078069, 'subsample': 0.9577304492446963, 'colsample_bytree': 0.9513090260647766}
Najlepsze RMSE CV XGBoost: 2.9746
Test RMSE XGBoost: 1.7313


lasso feature selection

In [ ]:
# ============================================
# IMPORTY
# ============================================
import joblib
import pandas as pd
import numpy as np
import sys
sys.path.append("..")
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import optuna
import xgboost as xgb

# ============================================
# ETAP 1: LASSO FEATURE SELECTION NA SAMPLE 10%
# ============================================
PATH = "../data/processed/features_data.csv"
df = pd.read_csv(PATH, parse_dates=["timestamp"])

df_sample = df.sample(frac=0.1, random_state=42)
#df_sample = df   # ta linijka zamiast powyzszej, zeby uzyc 100% danych

def preprocess(df):
    
    # Usuń kolumny stałe
    constant_cols = df.columns[df.nunique() <= 1]
    df = df.drop(columns=constant_cols)

    # Usuń kolumny z >40% braków
    na_cols = df.columns[df.isna().mean() > 0.4]
    df = df.drop(columns=na_cols)

    # Usuń timestamp i ID
    drop_cols = ['timestamp', 'campus_id', 'meter_id']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Tylko numeryczne
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df = df[numeric_cols]

    return df


df_clean = preprocess(df_sample)

y = df_clean['consumption']
X = df_clean.drop(columns=['consumption'])

lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.01, max_iter=5000))
])

lasso.fit(X, y)

coefs = lasso.named_steps['lasso'].coef_
selected_features = X.columns[coefs != 0].tolist()

pd.Series(selected_features).to_csv(
    "../data/processed/selected_features_lasso.csv",
    index=False
)

print("WYBRANE FEATURE’Y:", selected_features)


# ============================================
# ETAP 2: WCZYTANIE PEŁNYCH DANYCH POD FINALNY MODEL
# ============================================
df_full = pd.read_csv(PATH, parse_dates=["timestamp"])
df_full = preprocess(df_full)

df_full = df_full[selected_features + ['consumption']]

X_full = df_full.drop(columns=['consumption'])
y_full = df_full['consumption']

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)


# ============================================
# ETAP 3: OPTUNA – RIDGE
# ============================================
def objective_ridge(trial):
    alpha = trial.suggest_float("alpha", 0.0001, 10.0, log=True)

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=alpha))
    ])

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = mean_squared_error(y_test, preds) ** 0.5
    return rmse

study_ridge = optuna.create_study(direction="minimize")
study_ridge.optimize(objective_ridge, n_trials=30)

print("BEST RIDGE:", study_ridge.best_params)


# ============================================
# ETAP 4: OPTUNA – XGBOOST
# ============================================
def objective_xgb(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "eta": trial.suggest_float("eta", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
    }

    model = xgb.XGBRegressor(
        **params,
        n_estimators=300,
        tree_method="hist",
        eval_metric="rmse"
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds) ** 0.5

    return rmse

study_xgb = optuna.create_study(direction="minimize")
study_xgb.optimize(objective_xgb, n_trials=30)

print("BEST XGBOOST:", study_xgb.best_params)

joblib.dump(study_xgb.best_params, "models/xgb_best_params.pkl")
joblib.dump((X_train, X_test, y_train, y_test), "models/data_for_shap.pkl")



WYBRANE FEATURE’Y: ['gross_floor_area', 'room_area', 'capacity', 'is_holiday', 'is_semester', 'day_of_week', 'hour_sin', 'is_morning', 'consumption_lag_1h', 'consumption_lag_168h', 'consumption_rolling_mean_3h', 'consumption_rolling_std_3h', 'consumption_rolling_std_6h', 'consumption_rolling_mean_24h', 'consumption_per_sqm', 'consumption_per_person']


[I 2025-11-27 21:04:10,756] A new study created in memory with name: no-name-9cde84fc-ee86-497d-afe0-2b90ad4432f1
[I 2025-11-27 21:04:11,536] Trial 0 finished with value: 1.0737372771708742 and parameters: {'alpha': 0.008288608952356774}. Best is trial 0 with value: 1.0737372771708742.
[I 2025-11-27 21:04:12,318] Trial 1 finished with value: 1.0737372805276615 and parameters: {'alpha': 0.0038554069455761044}. Best is trial 0 with value: 1.0737372771708742.
[I 2025-11-27 21:04:12,943] Trial 2 finished with value: 1.073737283288763 and parameters: {'alpha': 0.00020954227526264306}. Best is trial 0 with value: 1.0737372771708742.
[I 2025-11-27 21:04:13,644] Trial 3 finished with value: 1.0737372833624172 and parameters: {'alpha': 0.00011229422803354081}. Best is trial 0 with value: 1.0737372771708742.
[I 2025-11-27 21:04:14,261] Trial 4 finished with value: 1.0737364292392533 and parameters: {'alpha': 1.1565816608944623}. Best is trial 4 with value: 1.0737364292392533.
[I 2025-11-27 21:04

BEST RIDGE: {'alpha': 9.353942993940331}


[I 2025-11-27 21:04:50,747] Trial 0 finished with value: 0.3026600762183675 and parameters: {'max_depth': 7, 'eta': 0.2776817060716482, 'subsample': 0.5559661685755065}. Best is trial 0 with value: 0.3026600762183675.
[I 2025-11-27 21:05:16,546] Trial 1 finished with value: 0.35103394618275596 and parameters: {'max_depth': 8, 'eta': 0.06973324972079119, 'subsample': 0.5132825195086652}. Best is trial 0 with value: 0.3026600762183675.
[I 2025-11-27 21:05:32,242] Trial 2 finished with value: 0.41325637673562027 and parameters: {'max_depth': 5, 'eta': 0.2626240225803721, 'subsample': 0.8986746794565863}. Best is trial 0 with value: 0.3026600762183675.
[I 2025-11-27 21:05:51,600] Trial 3 finished with value: 0.3591384473237131 and parameters: {'max_depth': 6, 'eta': 0.20105337545730728, 'subsample': 0.7783362046499702}. Best is trial 0 with value: 0.3026600762183675.
[I 2025-11-27 21:06:14,409] Trial 4 finished with value: 0.39277931933497706 and parameters: {'max_depth': 7, 'eta': 0.08623

BEST XGBOOST: {'max_depth': 9, 'eta': 0.21851888427337868, 'subsample': 0.7910114299500562}


['models/data_for_shap.pkl']

In [22]:
optuna.visualization.plot_optimization_history(study_xgb)

In [23]:
optuna.visualization.plot_slice(study_xgb)

In [24]:
optuna.visualization.plot_parallel_coordinate(study_xgb)